# SPIQ Knapsack Workflow

End-to-end SPIQ initialization on a small Knapsack instance, followed by two point-selection strategies for multi-start optimization.

In [ ]:
import warnings

import numpy as np
from qiskit.circuit.library import QAOAAnsatz
from qiskit_optimization.converters import QuadraticProgramToQubo

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from spiq.knapsack import generate_knapsack_instance
from spiq.qaoa import QAOASolver
from spiq.selection import fixed_interval_selection, k_gaps_selection

In [ ]:
N_ITEMS = 4
REPS = 2
N_GENS = 4
SEED = 7
NUM_SELECT = 3

np.random.seed(SEED)

prob = generate_knapsack_instance(num_items=N_ITEMS, seed=SEED)
qp = prob.to_quadratic_program()
qubo = QuadraticProgramToQubo().convert(qp)
cost_hamiltonian, offset = qubo.to_ising()

circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=REPS)
solver = QAOASolver(cost_hamiltonian, circuit, sim_device="CPU")
solver.err = None
solver.prepare_circuit()
exact_energy = solver.evaluate_exact_energy()

print(qp.prettyprint())
print(f"qubits={cost_hamiltonian.num_qubits}, offset={offset:.6f}")
print(f"exact energy={exact_energy:.6f}")

In [ ]:
solver.run_spiq(n_gens=N_GENS)

best_params = solver.best_spiq_gen_params[::-1]
best_fitness = solver.best_spiq_gen_fitness[::-1]
print(f"SPIQ best energy={solver.energy_best:.6f}")
print(f"candidate points={len(best_fitness)}")

In [ ]:
spaced_params, spaced_fitness = fixed_interval_selection(
    best_params, best_fitness, num_select=NUM_SELECT
)
for i, (params, energy) in enumerate(zip(spaced_params, spaced_fitness)):
    print(f"fixed interval {i + 1}: energy={energy:.6f}, params={params}")

In [ ]:
kgaps_result = k_gaps_selection(
    best_params,
    best_fitness,
    solver,
    num_select=NUM_SELECT,
    rng=np.random.default_rng(SEED),
)
if kgaps_result is None:
    print("k-gaps selection returned no points")
else:
    kgaps_params, kgaps_fitness, kgaps_grads = kgaps_result
    for i, (params, energy, grad) in enumerate(zip(kgaps_params, kgaps_fitness, kgaps_grads)):
        print(f"k-gaps {i + 1}: energy={energy:.6f}, grad_norm={grad:.6f}, params={params}")